# Delivery Delay Severity Analysis

## Objective

This notebook examines whether customer satisfaction declines as delivery delays become more severe.

The analysis uses the validated order-level dataset generated by `04_core_dataset.ipynb`.

## Main questions

1. How are orders distributed across different delivery-delay categories?
2. Does the average review score decline as delay days increase?
3. Does the proportion of low ratings increase with more severe delays?
4. Is the observed relationship economically meaningful for an e-commerce business?

The results describe associations and should not be interpreted as definitive causal effects.

In [1]:
from pathlib import Path
import pandas as pd
import duckdb

project_root = Path.cwd()

if not (project_root / "data" / "processed").exists():
    project_root = project_root.parent

core_dataset_path = (
    project_root
    / "data"
    / "processed"
    / "core_analysis_orders.csv"
)

core_orders = pd.read_csv(
    core_dataset_path
)

connection = duckdb.connect()
connection.register(
    "core_orders",
    core_orders
)

print("Core dataset file exists:", core_dataset_path.exists())
print("Core dataset shape:", core_orders.shape)
print(
    "Duplicated order_id rows:",
    core_orders["order_id"].duplicated().sum()
)

Core dataset file exists: True
Core dataset shape: (95824, 17)
Duplicated order_id rows: 0


## 1. Create Delivery-Delay Categories

Orders are divided into business-friendly delay categories to examine whether customer satisfaction changes as delivery performance worsens.

In [2]:
delay_group_counts = connection.execute("""
SELECT
    CASE
        WHEN delay_days <= -8 THEN 'Early by 8+ days'
        WHEN delay_days BETWEEN -7 AND -1 THEN 'Early by 1-7 days'
        WHEN delay_days = 0 THEN 'Delivered on estimated date'
        WHEN delay_days BETWEEN 1 AND 3 THEN 'Delayed by 1-3 days'
        WHEN delay_days BETWEEN 4 AND 7 THEN 'Delayed by 4-7 days'
        WHEN delay_days BETWEEN 8 AND 14 THEN 'Delayed by 8-14 days'
        WHEN delay_days >= 15 THEN 'Delayed by 15+ days'
    END AS delay_group,

    COUNT(*) AS order_count

FROM core_orders

GROUP BY delay_group

ORDER BY MIN(delay_days)
""").fetchdf()

display(delay_group_counts)

,delay_group,order_count
0,Early by 8+ days,70929
1,Early by 1-7 days,17234
2,Delivered on estimated date,1280
3,Delayed by 1-3 days,1852
4,Delayed by 4-7 days,1748
5,Delayed by 8-14 days,1446
6,Delayed by 15+ days,1335


In [3]:
delay_group_summary = connection.execute("""
SELECT
    CASE
        WHEN delay_days <= -8 THEN 'Early by 8+ days'
        WHEN delay_days BETWEEN -7 AND -1 THEN 'Early by 1-7 days'
        WHEN delay_days = 0 THEN 'Delivered on estimated date'
        WHEN delay_days BETWEEN 1 AND 3 THEN 'Delayed by 1-3 days'
        WHEN delay_days BETWEEN 4 AND 7 THEN 'Delayed by 4-7 days'
        WHEN delay_days BETWEEN 8 AND 14 THEN 'Delayed by 8-14 days'
        WHEN delay_days >= 15 THEN 'Delayed by 15+ days'
    END AS delay_group,

    COUNT(*) AS order_count,

ROUND(
    AVG(review_score),
    2
) AS average_review_score,

MEDIAN(review_score) AS median_review_score,

ROUND(
    100.0 * AVG(
        CASE
            WHEN review_score <= 2 THEN 1
            ELSE 0
        END
    ),
    2
) AS low_rating_percent

FROM core_orders

GROUP BY delay_group

ORDER BY MIN(delay_days)
""").fetchdf()

display(delay_group_summary)

,delay_group,order_count,average_review_score,median_review_score,low_rating_percent
0,Early by 8+ days,70929,4.32,5.0,8.97
1,Early by 1-7 days,17234,4.20,5.0,10.24
2,Delivered on estimated date,1280,4.03,5.0,12.42
3,Delayed by 1-3 days,1852,3.29,4.0,32.13
4,Delayed by 4-7 days,1748,2.10,1.0,67.68
5,Delayed by 8-14 days,1446,1.67,1.0,80.15
6,Delayed by 15+ days,1335,1.72,1.0,78.35


## 2. Interpretation of Delay Severity

The results indicate a strong but nonlinear negative association between delivery delays and customer satisfaction.

Orders delivered eight or more days early have an average review score of 4.32 and a low-rating rate of 8.97%. For orders delivered on the estimated date, the corresponding figures are 4.03 and 12.42%.

Customer satisfaction deteriorates sharply once an order becomes delayed. Orders delayed by one to three days have an average score of 3.29 and a low-rating rate of 32.13%. For delays of four to seven days, the average score falls to 2.10 and the low-rating rate rises to 67.68%.

The results suggest that businesses should reduce delivery delays generally and give particular priority to preventing delays from exceeding three days. However, determining the optimal allocation of operational resources would require additional information about intervention costs, feasibility and expected effectiveness.

These findings represent associations rather than definitive causal effects.